In [1]:
import sys
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Proje ana dizinine erişim sağlama (src klasörünü görebilmesi için)
sys.path.append(os.path.abspath(".."))

from src.models import ModelFactory
from src.evaluate import ModelEvaluator

# İşlenmiş veriyi okuyoruz
train_proc = pd.read_csv("../data/processed/train_processed.csv")

# Hedef değişken (Survived) ile bağımsız değişkenleri (X) ayırma
X = train_proc.drop(columns=['Survived'])
y = train_proc['Survived']

print("Veri Başarıyla Yüklendi!")
print(f"Girdi Matrisi Boyutu (X): {X.shape}")
print(f"Hedef Değişken (y) Dağılımı:\n{y.value_counts()}")

Veri Başarıyla Yüklendi!
Girdi Matrisi Boyutu (X): (891, 15)
Hedef Değişken (y) Dağılımı:
Survived
0    549
1    342
Name: count, dtype: int64


In [2]:
# KNN, SVM, Logistic Regression gibi modeller için veriyi standart ölçeklemeye sokuyoruz
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

# Model fabrikamızı ve değerlendirme sınıfımızı başlatıyoruz
factory = ModelFactory()
evaluator = ModelEvaluator()

baseline_models = factory.get_models()
param_grids = factory.get_param_grids()

all_results = []
trained_best_models = {}

print("Ölçekleme ve Nesne Kurulumları Tamamlandı.")

Ölçekleme ve Nesne Kurulumları Tamamlandı.


In [3]:
print("=== 1. BASELINE MODELLER EĞİTİLİYOR ===")

for name, model in baseline_models.items():
    # Ağaç tabanlı modeller (Decision Tree, Random Forest, Gradient Boosting) 
    # ölçekleme gerektirmediği için ham X verisini alır; diğerleri X_scaled alır.
    X_train = X if name in ["Decision Tree", "Random Forest", "Gradient Boosting"] else X_scaled
    
    res = evaluator.evaluate_model(model, X_train, y, model_name=name)
    all_results.append(res)
    
    print(f"[TAMAMLANDI] {name:20s} -> Val Accuracy: {res['Val Accuracy']:.4f} | F1-Score: {res['F1-Score']:.4f}")

=== 1. BASELINE MODELLER EĞİTİLİYOR ===
[TAMAMLANDI] Logistic Regression  -> Val Accuracy: 0.8283 | F1-Score: 0.7731
[TAMAMLANDI] Decision Tree        -> Val Accuracy: 0.7845 | F1-Score: 0.7224
[TAMAMLANDI] Random Forest        -> Val Accuracy: 0.8260 | F1-Score: 0.7690
[TAMAMLANDI] Gradient Boosting    -> Val Accuracy: 0.8417 | F1-Score: 0.7799
[TAMAMLANDI] KNN                  -> Val Accuracy: 0.8305 | F1-Score: 0.7729


c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The

[TAMAMLANDI] SVM                  -> Val Accuracy: 0.8283 | F1-Score: 0.7644
[TAMAMLANDI] Naive Bayes          -> Val Accuracy: 0.7957 | F1-Score: 0.7379


c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


[TAMAMLANDI] MLP (Neural Net)     -> Val Accuracy: 0.8238 | F1-Score: 0.7574


c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [4]:
print("=== 2. HYPERPARAMETER TUNING (GRIDSEARCHCV) UYGULANIYOR ===")

for name, model in baseline_models.items():
    X_train = X if name in ["Decision Tree", "Random Forest", "Gradient Boosting"] else X_scaled
    grid = param_grids[name]
    
    best_model, res = evaluator.tune_and_evaluate(model, grid, X_train, y, model_name=name)
    all_results.append(res)
    trained_best_models[name] = best_model
    
    print(f"[TUNED] {name:20s} -> Val Accuracy: {res['Val Accuracy']:.4f} | F1-Score: {res['F1-Score']:.4f}")

=== 2. HYPERPARAMETER TUNING (GRIDSEARCHCV) UYGULANIYOR ===
[TUNED] Logistic Regression  -> Val Accuracy: 0.8294 | F1-Score: 0.7748
[TUNED] Decision Tree        -> Val Accuracy: 0.8227 | F1-Score: 0.7565
[TUNED] Random Forest        -> Val Accuracy: 0.8384 | F1-Score: 0.7769
[TUNED] Gradient Boosting    -> Val Accuracy: 0.8451 | F1-Score: 0.7856
[TUNED] KNN                  -> Val Accuracy: 0.8305 | F1-Score: 0.7729


c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The

[TUNED] SVM                  -> Val Accuracy: 0.8283 | F1-Score: 0.7644
[TUNED] Naive Bayes          -> Val Accuracy: 0.7957 | F1-Score: 0.7379


c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\Alperen\Masaüstü\titanic_project\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged ye

[TUNED] MLP (Neural Net)     -> Val Accuracy: 0.8316 | F1-Score: 0.7672


In [ ]:
# Sonuçları DataFrame yapma
results_df = pd.DataFrame(all_results)

# outputs klasörü yoksa oluştur
os.makedirs("../outputs", exist_ok=True)
results_df.to_csv("../outputs/results_table.csv", index=False)

print("\n--- İŞLEM BAŞARIYLA TAMAMLANDI ---")
print("Sonuç tablosu 'outputs/results_table.csv' konumuna kaydedildi.\n")

print("🏆 En Başarılı 5 Model (F1-Score Sıralamasına Göre):")
print(results_df.sort_values(by="F1-Score", ascending=False)[['Model', 'Val Accuracy', 'F1-Score', 'ROC-AUC', 'Fit Time (sec)']].head(5))